# 02 · Daily Aggregation (M2)

Demonstrates the aggregation flow of `demandlens.data.aggregate` and actually runs it on a small file, then loads the official product `daily_all_categories.csv` (multi-category store, 7 months, batch-run locally by `scripts/build_daily.py`) for summary verification.

## Pipeline (corresponds to assignment 3.5)

Raw event CSV → chunked reading (1M rows/chunk) → parse time and event type → aggregate by date × category → daily Views / Cart / Purchase → verification statistics. The consistency invariants are covered by `tests/test_aggregate.py`.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
for cand in (ROOT, ROOT.parent, ROOT.parent.parent):
    if (cand / 'src' / 'demandlens').exists():
        ROOT = cand
        break
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import kagglehub
from demandlens.data.aggregate import aggregate_month

print('ready')

ready


## Run aggregation on the demo file

In [2]:
RAW = kagglehub.dataset_download(
    'mkechinov/ecommerce-events-history-in-cosmetics-shop',
    path='2019-Oct.csv')
daily, stats = aggregate_month(RAW, chunksize=500_000, progress_every=4)
print('raw rows:', f"{stats['raw_rows']:,}")
print('aggregatable rows:', f"{stats['aggregatable_rows']:,}")
print('contract event counts:', dict(stats['contract_kept']))
daily.head()

    2019-Oct.csv: processed 4 chunks / 2,000,000 rows


    2019-Oct.csv: processed 8 chunks / 4,000,000 rows


raw rows: 4,102,283
aggregatable rows: 3,340,173
contract event counts: {'view': 1862164, 'cart': 1232385, 'purchase': 245624}


event_type,date,category_id,category_code,views,cart,purchase
0,2019-10-01,1487580004832248652,None,364.0,510.0,31.0
1,2019-10-01,1487580004857414477,None,174.0,183.0,4.0
2,2019-10-01,1487580004882580302,None,156.0,109.0,24.0
3,2019-10-01,1487580004916134735,None,454.0,730.0,86.0
4,2019-10-01,1487580004983243602,None,7.0,3.0,0.0


## Aggregation consistency check (D2)

The count of each event type that entered aggregation must equal the sum of the daily-aggregated columns.

In [3]:
checks = {
    'view': (stats['contract_kept']['view'], int(daily['views'].sum())),
    'cart': (stats['contract_kept']['cart'], int(daily['cart'].sum())),
    'purchase': (stats['contract_kept']['purchase'], int(daily['purchase'].sum())),
}
pd.DataFrame([(k, a, b, b - a) for k, (a, b) in checks.items()],
             columns=['event', 'raw_kept', 'aggregated', 'diff'])

,event,raw_kept,aggregated,diff
0,view,1862164,1862164,0
1,cart,1232385,1232385,0
2,purchase,245624,245624,0


## Official product: multi-category store daily table

Load `data/processed/daily_all_categories.csv` below (if not yet generated, run `scripts/run_m2_pipeline.sh` first).

In [4]:
from demandlens.data.loader import CONTRACT_EVENTS

OUT = ROOT / 'data' / 'processed' / 'daily_all_categories.csv'
if OUT.exists():
    full = pd.read_csv(OUT, parse_dates=['date'])
    print('rows:', f'{len(full):,}')
    print('dates:', full['date'].min().date(), '..', full['date'].max().date())
    print('categories:', full['category_id'].nunique())
    print('daily purchase events total/mean:', int(full['purchase'].sum()),
          round(full['purchase'].mean(), 1))
    print('per-day purchase-event distribution:')
    per_day = full.groupby('date')['purchase'].sum()
    print(per_day.describe().round(1))
else:
    print('official product not yet generated (multi-month aggregation still running or not run)')

rows: 36,555
dates: 2019-10-01 .. 2019-11-30
categories: 691
daily purchase events total/mean: 1659788 45.4
per-day purchase-event distribution:
count        61.0
mean      27209.6
std       21726.4
min           0.0
25%       22458.0
50%       24305.0
75%       25385.0
max      185195.0
Name: purchase, dtype: float64


## Handoff notes

- This table does **not zero-fill and does not filter categories** (M2's product per contract 3.6).
- After M3 produces `daily_category.csv` through quality audit and category selection, downstream uses only the latter.
- Check details: `reports/tables/aggregation_check.csv`; report interpretation: `reports/aggregation_check_report.md`.